# 🌌 V50: THE MASTER UNIVERSE (Global Spatial-IDW Physics) 🌌\nBerdasarkan EDA mendalam, banjir murni dipicu oleh **Tanah Dalam yang Jenuh** dan **Area Beton**, bukan sekadar hujan sesaat. V50 membuang semua rute sungai yang rentan patah dan menggunakan Geostatistik murni (IDW Spatial Lag) dalam 1 Model Global XGBoost raksasa!

In [ ]:
# 1. MOUNT GOOGLE DRIVE & INSTALL LIBRARIES
!pip install -q xgboost optuna ipywidgets haversine

from google.colab import drive
import os
import warnings
warnings.filterwarnings('ignore')

print("Meminta Izin Akses Google Drive...")
drive.mount('/content/drive')

# UBAH BARIS INI SESUAI DENGAN FOLDER ANDA
PROJECT_PATH = '/content/drive/MyDrive/SSF-SSDS/'
os.chdir(PROJECT_PATH)
print(f"Bekerja di direktori: {os.getcwd()}")

In [ ]:
# 2. IMPORT LIBRARY & LOAD DATA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
from haversine import haversine, Unit
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

print("Memuat Dataset Utama...")
train_df = pd.read_csv('train.csv', parse_dates=['datetime'])
test_df = pd.read_csv('test.csv')
env_df = pd.read_csv('data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
sample_sub = pd.read_csv('sample_submission.csv')
coords_df = pd.read_csv('data_pendukung/koordinat_pos.csv')

# Parsing Test Set
test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]

print("Data berhasil dimuat!")

### TAHAP 1: MATRIKS GEOSPATIAL MURNI (Distance & Static Features)

In [ ]:
print("Membangun Matriks Jarak Haversine (Geostatistik)...")

# Hitung Jarak antar semua stasiun (Distance Matrix)
stations = coords_df['nama_pos'].values
dist_matrix = pd.DataFrame(index=stations, columns=stations, dtype=float)

for i, row_i in coords_df.iterrows():
    for j, row_j in coords_df.iterrows():
        if i == j:
            dist_matrix.loc[row_i['nama_pos'], row_j['nama_pos']] = 0.0
        else:
            p1 = (row_i['latitude'], row_i['longitude'])
            p2 = (row_j['latitude'], row_j['longitude'])
            dist_matrix.loc[row_i['nama_pos'], row_j['nama_pos']] = haversine(p1, p2, unit=Unit.KILOMETERS)

# Data Statis (Beton, HydroRIVERS)
static_mapping = env_df.groupby('nama_pos').first()[['built_surface_m2']].to_dict('index')

# Asumsi Mapping HydroRIVERS (Jika Anda punya file hydro_mapping.json, bisa diload di sini)
# Kita akan menggunakan elevasi dan luas area sebagai proksi statis.
elevations = train_df.groupby('nama_pos')['tma_mdpl'].mean().to_dict()

print("Matriks Jarak dan Data Statis siap!")

### TAHAP 2: FISIKA ALAM & INVERSE DISTANCE WEIGHTING (IDW)

In [ ]:
print("Mengekstraksi Fisika Tanah Dalam & Cuaca Ekstrem...")
env_df = env_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
grouped = env_df.groupby('nama_pos')

# Fisika Cuaca Lokal
env_df['rain_sum_72h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(72, min_periods=1).sum())
env_df['rain_sum_168h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(168, min_periods=1).sum())
env_df['rain_ewm_48h'] = grouped['rainfall_mm'].transform(lambda x: x.ewm(halflife=48, ignore_na=True).mean())

# FISIKA TANAH DALAM (The Hidden Truth)
if 'soil_moisture_100_255cm' in env_df.columns:
    # Jenuhnya tanah dalam memicu banjir tak terkendali
    env_df['deep_soil_ewm_168h'] = grouped['soil_moisture_100_255cm'].transform(lambda x: x.ewm(halflife=168, ignore_na=True).mean())
    env_df['soil_0_7cm_sum_72h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(72, min_periods=1).sum())
else:
    print("Warning: Data tanah dalam tidak ditemukan!")

print("Menghitung Inverse Distance Weighting (IDW) Spatial Lag...")
# Kita akan membuat DataFrame Pivot untuk Curah Hujan 168h agar perhitungan IDW vektorisasi bisa dilakukan
pivot_rain = env_df.pivot(index='datetime', columns='nama_pos', values='rain_sum_168h')

idw_features = []
# Hindari pembagian dengan nol dengan menambahkan epislon kecil pada jarak
epsilon = 1.0 

for pos in stations:
    weights = 1.0 / (dist_matrix.loc[pos] + epsilon)
    weights[pos] = 0.0 # Jangan hitung diri sendiri
    
    # Hitung bobot normalisasi
    weight_sum = weights.sum()
    
    # Kalikan curah hujan stasiun lain dengan bobot jaraknya
    idw_rain = (pivot_rain * weights).sum(axis=1) / weight_sum
    
    idw_df = pd.DataFrame({
        'datetime': pivot_rain.index,
        'nama_pos': pos,
        'spatial_idw_rain_168h': idw_rain.values
    })
    idw_features.append(idw_df)

spatial_idw_df = pd.concat(idw_features, ignore_index=True)
env_df = pd.merge(env_df, spatial_idw_df, on=['datetime', 'nama_pos'], how='left')

print("Fisika Alam & Spatial IDW selesai!")

### TAHAP 3: MERAKIT DATASET GLOBAL

In [ ]:
def prepare_global_dataset(base_df, env_feat, historical_tma_df):
    df = base_df.copy()
    
    # 1. Fitur Siklik & Waktu
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    df['year'] = df['datetime'].dt.year
    df['year_month'] = df['year'].astype(str) + "_" + df['month'].astype(str)
    
    # 2. Menggabungkan Lingkungan & Geostatistik
    keep_env = ['datetime', 'nama_pos', 'rain_sum_72h', 'rain_sum_168h', 'rain_ewm_48h', 
                'deep_soil_ewm_168h', 'soil_0_7cm_sum_72h', 'spatial_idw_rain_168h', 
                'temperature_c', 'built_surface_m2']
    df = pd.merge(df, env_feat[keep_env], on=['datetime', 'nama_pos'], how='left')
    
    # 3. Fitur Statis Global
    df['elevation_proxy'] = df['nama_pos'].map(elevations)
    
    # 4. Memori Stasiun Sendiri (Jangkar)
    historical_map = historical_tma_df.set_index(['nama_pos', 'datetime'])['tma_mdpl'].to_dict()
    df['tma_lag_365d'] = df.apply(lambda row: historical_map.get((row['nama_pos'], row['datetime'] - pd.DateOffset(years=1)), np.nan), axis=1)
    
    last_known_map = historical_tma_df.dropna(subset=['tma_mdpl']).sort_values('datetime').groupby('nama_pos').tail(1).set_index('nama_pos')['tma_mdpl'].to_dict()
    df['last_known_tma'] = df['nama_pos'].map(last_known_map)
    
    # 5. Konversi Nama Pos menjadi Kategori Native XGBoost (SANGAT KRUSIAL)
    df['nama_pos_cat'] = df['nama_pos'].astype('category')
    
    return df

historical_tma = train_df[['datetime', 'nama_pos', 'tma_mdpl']].copy()

print("Memproses Train Set Global (84.396 baris)...")
train_full = prepare_global_dataset(train_df, env_df, historical_tma)
print("Memproses Test Set Global...")
test_full = prepare_global_dataset(test_parsed, env_df, historical_tma)

# Forward Fill Memori Musiman
for st in train_full['nama_pos'].unique():
    mask_tr = train_full['nama_pos'] == st
    train_full.loc[mask_tr, 'tma_lag_365d'] = train_full.loc[mask_tr, 'tma_lag_365d'].fillna(train_full.loc[mask_tr, 'last_known_tma'])
    mask_te = test_full['nama_pos'] == st
    test_full.loc[mask_te, 'tma_lag_365d'] = test_full.loc[mask_te, 'tma_lag_365d'].fillna(test_full.loc[mask_te, 'last_known_tma'])

train_full = train_full.dropna(subset=['tma_mdpl']).reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id', 'year', 'year_month']
features = [c for c in train_full.columns if c not in drop_cols]
print(f"Total Fitur Global V50: {len(features)}")
print(features)

### TAHAP 4: PELATIHAN 1 MODEL RAKSASA (Tweedie XGBoost)

In [ ]:
print("Mempersiapkan Global Training (GroupKFold by Year-Month)...")

X_global = train_full[features]
y_global = train_full['tma_mdpl']
groups = train_full['year_month'] # Agar validasi tidak membocorkan cuaca di bulan yang sama

def objective(trial):
    params = {
        'objective': 'reg:tweedie', 
        'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.2, 1.8),
        'eval_metric': 'rmse', 
        'tree_method': 'hist',
        'enable_categorical': True, # Wajib untuk Global Model
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'n_estimators': 600,
        'random_state': 42
    }
    
    # Validasi Cepat dengan 1 Fold (misal bulan terakhir) untuk Optuna
    gkf = GroupKFold(n_splits=4)
    train_idx, val_idx = next(gkf.split(X_global, y_global, groups=groups))
    
    X_tr, y_tr = X_global.iloc[train_idx], np.clip(y_global.iloc[train_idx], 0.0001, None)
    X_va, y_va = X_global.iloc[val_idx], np.clip(y_global.iloc[val_idx], 0.0001, None)
    
    model = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    preds = model.predict(X_va)
    return np.sqrt(mean_squared_error(y_va, preds))

print("Mencari Hyperparameter Terbaik dengan Optuna (20 Trials)...")
optuna.logging.set_verbosity(optuna.logging.ERROR)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

best_params = study.best_params
best_params['objective'] = 'reg:tweedie'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'
best_params['enable_categorical'] = True
best_params['n_estimators'] = 800
best_params['random_state'] = 42

print(f"\n🏆 Best Single-Fold RMSE: {study.best_value:.4f}")
print("Melatih Model Final dengan Seluruh Data Train...")
model_final = xgb.XGBRegressor(**best_params)
model_final.fit(X_global, np.clip(y_global, 0.0001, None), verbose=50)
print("Selesai!")

### TAHAP 5: PEMBEDAHAN OTAK GLOBAL & SUBMISSION

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
imp_df = pd.DataFrame({
    'Features': features, 
    'Importance score': model_final.feature_importances_
}).sort_values(by='Importance score', ascending=False)

sns.barplot(x='Importance score', y='Features', data=imp_df, ax=ax, palette='viridis')
ax.set_title('Global Feature Importance (Bukti Fisika Alam)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Mencetak Prediksi untuk 21.780 baris Test Set...")
X_test = test_full[features]
test_full['tma_mdpl'] = model_final.predict(X_test)

sub_pred = test_full[['id', 'tma_mdpl']].copy()
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')

# Forward fill pengamanan
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

output_file = 'submission_V50_Master_Universe.csv'
sub.to_csv(output_file, index=False)
print(f"✅ Selesai! File '{output_file}' siap disubmit. The New Universe is Alive!")